# 🌲 AdaBoost — Solutions Notebook

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~45 mins  
**Complete, verified reference implementation.**

---


## 🎯 Section 1: Overview

Adaptive Boosting sequentially trains decision stumps, updating instance weights to focus on hard misclassified samples.

### Weight Update:
$$w_i^{(t+1)} = w_i^{(t)} \exp(-\alpha_t y_i h_t(x_i))$$
where $\alpha_t = \frac{1}{2} \ln\frac{1-\epsilon_t}{\epsilon_t}$.


## 🔧 Section 2: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

In [ ]:
from sklearn.tree import DecisionTreeClassifier

class AdaBoostFromScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.stumps = []
        
    def fit(self, X, y):
        # Target must be -1 or +1
        y_binary = np.where(y <= 0, -1, 1)
        n_samples = X.shape[0]
        w = np.full(n_samples, 1.0 / n_samples)
        
        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y_binary, sample_weight=w)
            predictions = stump.predict(X)
            
            # Weighted error
            misclassified = (y_binary != predictions)
            error = np.sum(w[misclassified]) / np.sum(w)
            error = np.clip(error, 1e-10, 1 - 1e-10)
            
            alpha = 0.5 * np.log((1.0 - error) / error)
            
            # Update weights
            w *= np.exp(-alpha * y_binary * predictions)
            w /= np.sum(w)
            
            self.stumps.append(stump)
            self.alphas.append(alpha)
        return self

    def predict(self, X):
        stump_preds = np.array([alpha * stump.predict(X) for stump, alpha in zip(self.stumps, self.alphas)])
        raw_output = np.sum(stump_preds, axis=0)
        return np.where(raw_output >= 0, 1, 0)


In [ ]:
X = np.random.rand(100, 4)
y = np.random.randint(0, 2, 100)
ada = AdaBoostFromScratch(n_estimators=10)
ada.fit(X, y)
print('AdaBoost preds:', ada.predict(X)[:5])


## 📦 Section 3: Library Implementation


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
adaboost = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)
adaboost.fit(X_train, y_train)


## ❓ Section 4: Interview Questions


### Q1: What happens if a weak learner in AdaBoost has an error rate > 0.5?
**Answer**: In binary classification, an error rate > 0.5 is worse than random guessing. Alpha becomes negative, effectively inverting the weak learner's predictions.


### Q2: Is AdaBoost sensitive to noise and outliers?
**Answer**: Yes. Misclassified outliers receive exponentially higher weights in successive iterations, forcing subsequent weak learners to overfit on noise.
